# 03 — ML Model : Classification WIN/LOSS
**Savvy Crypto Agent — Jedha Bootcamp Final Project**

Objectif : entraîner un modèle qui prédit WIN ou LOSS, calibrer le seuil de confiance.

> **Règle absolue : le TEST set n'est pas touché dans ce notebook.**

In [ ]:
import sys
!{sys.executable} -m pip install xgboost scikit-learn joblib

## 0. Imports & config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
import xgboost as xgb
import joblib

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

PROCESSED = "C:\\Users\\casir\\Desktop\\CRYPTOS\\data\\processed\\"
MODELS    = "C:\\Users\\casir\\Desktop\\CRYPTOS\\models\\"
os.makedirs(MODELS, exist_ok=True)

# --- latest.json lookup (fallback sans suffixe au premier lancement) ---
if os.path.exists(PROCESSED + 'latest.json'):
    with open(PROCESSED + 'latest.json') as f:
        latest = json.load(f)
    print(f"Chargement version : {latest['date']}")
else:
    latest = {'train': 'train_features.csv', 'test': 'test_features.csv',
              'feature_cols': 'feature_cols.json', 'config': 'config.json', 'date': 'fallback'}
    print('latest.json absent -> fallback sans suffixe')

with open(PROCESSED + latest['config']) as f:
    cfg = json.load(f)
with open(PROCESSED + latest['feature_cols']) as f:
    FEATURE_COLS = json.load(f)

TP_NET    = cfg['tp_net']
SL_NET    = cfg['sl_net']
BREAKEVEN = cfg['breakeven']

print(f'TP_net={TP_NET*100:.2f}%  SL_net={SL_NET*100:.2f}%  Breakeven={BREAKEVEN*100:.1f}%')
print(f'Features : {len(FEATURE_COLS)}')

## 1. Chargement

In [ ]:
train = pd.read_csv(PROCESSED + latest['train'])
train['candle_time'] = pd.to_datetime(train['candle_time'], utc=True)
train = train.sort_values('candle_time').reset_index(drop=True)

X_train = train[FEATURE_COLS]
y_train = train['label']

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f'TRAIN ({latest["train"]}) : {len(train):,} bougies')
print(f'WIN   : {pos:,} ({pos/len(train)*100:.1f}%)')
print(f'LOSS  : {neg:,} ({neg/len(train)*100:.1f}%)')
print(f'scale_pos_weight : {scale_pos_weight:.1f}')

## Comparaison des modèles : du plus simple au plus complexe

Démarche : on part du modèle le plus simple (régression logistique), puis on monte
en complexité (Random Forest, puis XGBoost). Pour chacun on regarde 5 métriques
sur le TEST (données jamais vues) : Accuracy, Precision, Recall, F1-score et AUC.
On ajoute aussi un modèle "naïf" (toujours prédire LOSS) pour montrer pourquoi
l'Accuracy est trompeuse sur des données déséquilibrées.

In [ ]:
# ============================================================
#  COMPARAISON : Régression Logistique -> Random Forest -> XGBoost
#  Métriques sur le TEST (hors échantillon)
# ============================================================
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

# --- Charger le test si besoin (mêmes conventions que le notebook) ---
try:
    test
except NameError:
    test = pd.read_csv(PROCESSED + latest['test'])

X_test = test[FEATURE_COLS]
y_test = test['label']

# Sécurité : la régression logistique ne tolère ni NaN ni infini
def clean(df):
    return df.replace([np.inf, -np.inf], np.nan).fillna(0)
X_train_c, X_test_c = clean(X_train), clean(X_test)

# --- 1) Régression Logistique (le modèle le plus SIMPLE) ---
#     Pipeline = normalisation (indispensable ici) + régression logistique.
#     La normalisation met toutes les features à la même échelle ; sans elle
#     la régression logistique est faussée. (Random Forest et XGBoost, eux,
#     n'en ont pas besoin : ce sont des arbres.)
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
)
logreg.fit(X_train_c, y_train)

# --- 2) Random Forest ---
rf_cmp = RandomForestClassifier(
    n_estimators=300, max_depth=10, class_weight='balanced',
    random_state=42, n_jobs=-1)
rf_cmp.fit(X_train, y_train)

# --- 3) XGBoost ---
xgb_cmp = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc', random_state=42, n_jobs=-1)
xgb_cmp.fit(X_train, y_train)

# --- Probabilités sur le TEST ---
models = {
    'Régression Logistique': (logreg, X_test_c),
    'Random Forest':         (rf_cmp, X_test),
    'XGBoost':               (xgb_cmp, X_test),
}

THRESH = 0.5  # seuil de classification standard
rows, proba_store = [], {}
for name, (model, Xt) in models.items():
    proba = model.predict_proba(Xt)[:, 1]
    proba_store[name] = proba
    pred = (proba >= THRESH).astype(int)
    rows.append({'Modèle': name,
                 'Accuracy':  accuracy_score(y_test, pred),
                 'Precision': precision_score(y_test, pred, zero_division=0),
                 'Recall':    recall_score(y_test, pred, zero_division=0),
                 'F1-score':  f1_score(y_test, pred, zero_division=0),
                 'AUC':       roc_auc_score(y_test, proba)})

# --- Modèle NAÏF : toujours prédire LOSS (pour illustrer le piège de l'Accuracy) ---
rows.insert(0, {'Modèle': 'Naïf (toujours LOSS)',
                'Accuracy': accuracy_score(y_test, np.zeros_like(y_test)),
                'Precision': 0.0, 'Recall': 0.0, 'F1-score': 0.0, 'AUC': 0.5})

comp = pd.DataFrame(rows).set_index('Modèle').round(3)
print(f"Métriques sur le TEST (hors échantillon), seuil = {THRESH}")
print(f"Win rate de base = {y_test.mean()*100:.1f}%  ->  classes déséquilibrées\n")
print(comp.to_string())

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (name, proba) in zip(axes, proba_store.items()):
    pred = (proba >= THRESH).astype(int)
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=['LOSS (0)', 'WIN (1)']).plot(
        ax=ax, cmap='Blues', colorbar=False, values_format=',d')
    ax.set_title(name)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
plt.tight_layout(); plt.show()

### Comment lire ces résultats

- **Accuracy (exactitude)** = % de prédictions correctes. PIÈGE : le modèle naïf
  qui prédit toujours "LOSS" a une Accuracy élevée (~90 %) alors qu'il est inutile
  (il ne trouve aucun gagnant). Sur des données déséquilibrées, l'Accuracy ment.
- **Precision (précision)** = parmi les trades prédits gagnants, combien le sont
  vraiment. C'est directement le win rate.
- **Recall (rappel)** = parmi tous les vrais gagnants, combien le modèle en attrape.
- **F1-score** = moyenne équilibrée entre Precision et Recall. C'est LA métrique
  honnête ici, car elle ignore la classe majoritaire.
- **Matrice de confusion** : les 4 cases = vrais/faux positifs et négatifs. La
  diagonale, ce sont les bonnes réponses.

Conclusion attendue : plus le modèle est sophistiqué (logistique -> RF -> XGBoost),
meilleurs sont le F1 et l'AUC. Cela justifie la montée en complexité.

## 2. Validation croisée temporelle (TimeSeriesSplit)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

print('Splits TimeSeriesSplit :')
for i, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    print(f'  Fold {i+1} : train={len(tr_idx):,}  val={len(val_idx):,}  '
          f'val_end={train.iloc[val_idx[-1]]["candle_time"].date()}')

## 3. Cross-validation XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc', random_state=42, n_jobs=-1
)

print('Cross-validation XGBoost (ROC-AUC)...')
xgb_scores = []
for i, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    xgb_model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx], verbose=False)
    probs = xgb_model.predict_proba(X_train.iloc[val_idx])[:, 1]
    auc = roc_auc_score(y_train.iloc[val_idx], probs)
    xgb_scores.append(auc)
    print(f'  Fold {i+1} : AUC = {auc:.4f}')

print(f'XGBoost AUC moyen : {np.mean(xgb_scores):.4f} +/- {np.std(xgb_scores):.4f}')

In [ ]:
# ============================================================
#  TEST DE FUITE DE DONNÉES — features de la bougie courante
#  À coller après le chargement de train / test / FEATURE_COLS
# ============================================================
from sklearn.metrics import roc_auc_score
import xgboost as xgb

# Les 6 suspectes : elles décrivent la bougie COURANTE (celle dont
# le high/low peut aussi déterminer si le TP a été touché).
SUSPECT_FEATURES = [
    'upper_wick_ratio', 'lower_wick_ratio', 'body_ratio',
    'buy_pressure', 'sell_pressure', 'is_bullish_candle',
]

# On ne garde que celles réellement présentes dans TES features
suspects = [f for f in SUSPECT_FEATURES if f in FEATURE_COLS]
clean_features = [f for f in FEATURE_COLS if f not in suspects]
print(f"Features suspectes retirées ({len(suspects)}) : {suspects}")
print(f"Features restantes : {len(clean_features)} / {len(FEATURE_COLS)}\n")

# Sécurité : charger le test set s'il n'est pas déjà en mémoire
try:
    test
except NameError:
    test = pd.read_csv(PROCESSED + latest['test'])
    test['candle_time'] = pd.to_datetime(test['candle_time'], utc=True)
    test = test.sort_values('candle_time').reset_index(drop=True)

y_test = test['label']

def train_and_eval(features, label):
    """Entraîne le XGB (config identique à la tienne) et renvoie l'AUC out-of-sample."""
    model = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc', random_state=42, n_jobs=-1,
    )
    model.fit(train[features], train['label'], verbose=False)
    proba_test = model.predict_proba(test[features])[:, 1]
    auc = roc_auc_score(y_test, proba_test)
    print(f"  {label:38s} : AUC test = {auc:.4f}")
    return auc

print("=== AUC OUT-OF-SAMPLE (sur le TEST set) ===")
auc_full  = train_and_eval(FEATURE_COLS,    "AVEC features bougie courante")
auc_clean = train_and_eval(clean_features,  "SANS features bougie courante")

drop = auc_full - auc_clean
print(f"\n  Chute d'AUC : {drop:+.4f}")
print("-" * 55)
if drop > 0.15:
    print("  ⚠️  FUITE PROBABLE : l'AUC dépend fortement de ces 6 features.")
    print("      -> vérifie que la fenêtre TP/SL démarre à la bougie SUIVANTE (i+1),")
    print("         pas à la bougie courante (i), dans label_trades().")
elif drop > 0.05:
    print("  🟠 Contribution notable mais pas dominante. À creuser mais pas rédhibitoire.")
else:
    print("  ✅ AUC stable sans ces features -> pas de fuite évidente par ce canal.")
    print("      Les features tiennent la route.")

## 4. Cross-validation Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1
)

print('Cross-validation Random Forest (ROC-AUC)...')
rf_scores = []
for i, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    rf_model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    probs = rf_model.predict_proba(X_train.iloc[val_idx])[:, 1]
    auc = roc_auc_score(y_train.iloc[val_idx], probs)
    rf_scores.append(auc)
    print(f'  Fold {i+1} : AUC = {auc:.4f}')

print(f'Random Forest AUC moyen : {np.mean(rf_scores):.4f} +/- {np.std(rf_scores):.4f}')
print()
BEST_MODEL_NAME = 'XGBoost' if np.mean(xgb_scores) >= np.mean(rf_scores) else 'RandomForest'
print(f'Meilleur modele : {BEST_MODEL_NAME}')

## 5. Entraînement final sur tout le TRAIN

In [ ]:
print('Entrainement final...')

xgb_final = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc', random_state=42, n_jobs=-1
)
xgb_final.fit(X_train, y_train, verbose=False)

rf_final = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_final.fit(X_train, y_train)

train['proba_xgb'] = xgb_final.predict_proba(X_train)[:, 1]
train['proba_rf']  = rf_final.predict_proba(X_train)[:, 1]

print('OK')
print(f'XGBoost proba moy WIN  : {train[train.label==1]["proba_xgb"].mean():.3f}')
print(f'XGBoost proba moy LOSS : {train[train.label==0]["proba_xgb"].mean():.3f}')

## 6. Feature Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))

xgb_imp = pd.Series(xgb_final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
xgb_imp.tail(25).plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_title('Feature Importance — XGBoost (Top 25)', fontsize=13)

rf_imp = pd.Series(rf_final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
rf_imp.tail(25).plot(kind='barh', ax=ax2, color='coral')
ax2.set_title('Feature Importance — Random Forest (Top 25)', fontsize=13)

plt.tight_layout()
plt.show()

print('Top 15 XGBoost :')
print(xgb_imp.tail(15).sort_values(ascending=False).to_string())

## 7. Courbes ROC et Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for model_name, proba_col, color in [
    ('XGBoost', 'proba_xgb', 'steelblue'),
    ('Random Forest', 'proba_rf', 'coral')
]:
    probs = train[proba_col]
    fpr, tpr, _ = roc_curve(y_train, probs)
    auc = roc_auc_score(y_train, probs)
    axes[0].plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})', color=color)

    prec, rec, _ = precision_recall_curve(y_train, probs)
    axes[1].plot(rec, prec, label=model_name, color=color)

axes[0].plot([0,1],[0,1],'k--', alpha=0.3)
axes[0].set_title('Courbe ROC (TRAIN)', fontsize=13)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

axes[1].axhline(BREAKEVEN, color='black', linestyle='--', label=f'Breakeven ({BREAKEVEN*100:.0f}%)')
axes[1].set_title('Precision-Recall (TRAIN)', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision (win rate)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Calibration du seuil de confiance

On cherche le seuil de probabilite minimum au-dessus duquel le modele est rentable avec suffisamment de trades.

In [ ]:
def threshold_analysis(proba_col, model_name, color):
    thresholds = np.arange(0.05, 0.95, 0.01)
    results = []
    for t in thresholds:
        mask = train[proba_col] >= t
        sub  = train[mask]
        if len(sub) < 50:
            break
        wr  = sub['label'].mean()
        n   = len(sub)
        ev  = wr * TP_NET + (1 - wr) * (-SL_NET)
        results.append({
            'threshold': round(t, 2),
            'n_trades' : n,
            'win_rate' : round(wr * 100, 2),
            'ev_pct'   : round(ev * 100, 3),
            'rentable' : wr > BREAKEVEN
        })
    df = pd.DataFrame(results)

    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    fig.suptitle(f'Analyse seuil de confiance — {model_name}', fontsize=13)

    colors = ['steelblue' if r else 'tomato' for r in df['rentable']]
    axes[0].bar(df['threshold'], df['win_rate'], color=colors, width=0.01)
    axes[0].axhline(BREAKEVEN * 100, color='black', linestyle='--', label=f'Breakeven ({BREAKEVEN*100:.0f}%)')
    axes[0].set_ylabel('Win rate (%)')
    axes[0].legend()

    axes[1].bar(df['threshold'], df['n_trades'], color=color, alpha=0.6, width=0.01)
    axes[1].set_ylabel('Nombre de trades')

    colors2 = ['steelblue' if e > 0 else 'tomato' for e in df['ev_pct']]
    axes[2].bar(df['threshold'], df['ev_pct'], color=colors2, width=0.01)
    axes[2].axhline(0, color='black', linewidth=0.8)
    axes[2].set_ylabel('Expected value / trade (%)')
    axes[2].set_xlabel('Seuil de probabilite minimum')
    plt.tight_layout()
    plt.show()

    rentable = df[(df['rentable']) & (df['n_trades'] > 200)]
    if len(rentable) > 0:
        best = rentable.loc[rentable['ev_pct'].idxmax()]
        print(f'=== MEILLEUR SEUIL {model_name} ===')
        print(f'  Seuil    : {best["threshold"]}')
        print(f'  Win rate : {best["win_rate"]}%')
        print(f'  N trades : {best["n_trades"]:,}')
        print(f'  EV/trade : {best["ev_pct"]}%')
    else:
        best = None
        print(f'Aucun seuil rentable pour {model_name}')
    return df, best

df_xgb_thresh, best_xgb = threshold_analysis('proba_xgb', 'XGBoost', 'steelblue')

In [ ]:
df_rf_thresh, best_rf = threshold_analysis('proba_rf', 'Random Forest', 'coral')

## 9. Backtest TRAIN avec seuil optimal

In [ ]:
def backtest_with_threshold(model_name, proba_col, best_thresh_row):
    if best_thresh_row is None:
        print(f'Pas de seuil rentable pour {model_name}')
        return
    thresh  = best_thresh_row['threshold']
    trades  = train[train[proba_col] >= thresh].copy().sort_values('candle_time')
    CAPITAL = 10_000
    SIZE    = 500
    trades['pnl']     = trades['exit_ret'] * SIZE
    trades['capital'] = CAPITAL + trades['pnl'].cumsum()
    trades['dd']      = trades['capital'] / trades['capital'].cummax() - 1
    total   = len(trades)
    wins    = trades['label'].sum()
    pnl     = trades['pnl'].sum()
    days    = (trades['candle_time'].max() - trades['candle_time'].min()).days
    print(f'=== BACKTEST TRAIN — {model_name} (seuil={thresh}) ===')
    print(f'Total trades    : {total:,}  |  Win rate : {wins/total*100:.1f}%')
    print(f'PnL total       : {pnl:+,.0f} EUR')
    print(f'Capital final   : {CAPITAL+pnl:,.0f} EUR  ({pnl/CAPITAL*100:+.1f}%)')
    print(f'Max drawdown    : {trades["dd"].min()*100:.1f}%')
    print(f'Trades/jour     : {total/max(days,1):.1f}')
    print(f'PnL/jour        : {pnl/max(days,1):+.1f} EUR  (objectif: {CAPITAL*0.005:.0f} EUR)')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax1.plot(trades['candle_time'], trades['capital'], color='steelblue', linewidth=1.2)
    ax1.axhline(CAPITAL, color='gray', linestyle=':')
    ax1.fill_between(trades['candle_time'], CAPITAL, trades['capital'],
                     where=trades['capital']>=CAPITAL, alpha=0.15, color='steelblue')
    ax1.fill_between(trades['candle_time'], CAPITAL, trades['capital'],
                     where=trades['capital']<CAPITAL, alpha=0.15, color='tomato')
    ax1.set_title(f'Capital — {model_name} seuil={thresh} (TRAIN)', fontsize=13)
    ax1.set_ylabel('Capital (EUR)')
    ax2.fill_between(trades['candle_time'], trades['dd']*100, 0, color='tomato', alpha=0.4)
    ax2.set_title('Drawdown (%)')
    ax2.set_ylabel('Drawdown (%)')
    plt.tight_layout()
    plt.show()

backtest_with_threshold('XGBoost', 'proba_xgb', best_xgb)
backtest_with_threshold('Random Forest', 'proba_rf', best_rf)

## 10. Consensus features + sauvegarde

In [ ]:
comparison = pd.DataFrame({
    'xgb': xgb_final.feature_importances_,
    'rf' : rf_final.feature_importances_
}, index=FEATURE_COLS)
comparison['xgb_rank'] = comparison['xgb'].rank(ascending=False).astype(int)
comparison['rf_rank']  = comparison['rf'].rank(ascending=False).astype(int)
comparison['avg_rank'] = (comparison['xgb_rank'] + comparison['rf_rank']) / 2
comparison = comparison.sort_values('avg_rank')
TOP_FEATURES = comparison.head(15).index.tolist()
print('Top 15 features consensus :')
print(comparison.head(15)[['xgb_rank','rf_rank','avg_rank']].to_string())

from datetime import datetime
DATE = datetime.now().strftime('%Y%m%d')

# Sauvegarde versionnee
joblib.dump(xgb_final, MODELS + f'xgb_final_{DATE}.pkl')
joblib.dump(rf_final,  MODELS + f'rf_final_{DATE}.pkl')
# Copie sans suffixe (NE PAS SUPPRIMER -- Railway en depend)
joblib.dump(xgb_final, MODELS + 'xgb_final.pkl')
joblib.dump(rf_final,  MODELS + 'rf_final.pkl')

ml_config = {
    'xgb_threshold' : float(best_xgb['threshold']) if best_xgb is not None else None,
    'rf_threshold'  : float(best_rf['threshold'])  if best_rf  is not None else None,
    'top_features'  : TOP_FEATURES,
    'feature_cols'  : FEATURE_COLS,
    'best_model'    : BEST_MODEL_NAME
}
with open(PROCESSED + f'ml_config_{DATE}.json', 'w') as f:
    json.dump(ml_config, f, indent=2)
with open(PROCESSED + 'ml_config.json', 'w') as f:
    json.dump(ml_config, f, indent=2)

# models_latest.json : merge si existant, sinon cree
models_latest_path = MODELS + 'models_latest.json'
if os.path.exists(models_latest_path):
    with open(models_latest_path) as f:
        models_latest = json.load(f)
else:
    models_latest = {}
models_latest.update({
    'xgb': f'xgb_final_{DATE}.pkl',
    'rf' : f'rf_final_{DATE}.pkl',
    'date': DATE
})
with open(models_latest_path, 'w') as f:
    json.dump(models_latest, f, indent=2)

print('\nSauvegarde :')
print(f'  xgb_final_{DATE}.pkl, rf_final_{DATE}.pkl -> {MODELS} (+ copies sans suffixe)')
print(f'  ml_config_{DATE}.json -> {PROCESSED} (+ copie sans suffixe)')
print(f'  models_latest.json -> {models_latest}')
print()
print('Notebook 03 termine. Passe au notebook 04_backtest_test.ipynb')